## Task #1 - Web Scraping
**Please run the code one by one (cell by cell)**

## Import libraries
**Import Python libraries needed for web scraping and data processing**

In [14]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

## Setup Browser 
**Initialize the Selenium Chrome browser for automated web scraping**

In [15]:
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # Uncomment if you don't want browser to open

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
print("Browser started successfully!")

Browser started successfully!


### Before running the code below, please copy and paste the URL https://www.cityu.edu.hk/hktech300/start-ups/all-start-ups to the browser that was started in code above.

## Scrape 40 pages from the website
**Load and go through all 40 pages to extract Company Name, CityU URL, Company Website and Email**

In [16]:
BASE_URL = "https://www.cityu.edu.hk/hktech300/start-ups/all-start-ups"
all_companies = []

for page in range(1, 41):  # Pages 1 to 40
    print(f"Scraping page {page} of 40...")

    driver.get(f"{BASE_URL}?page={page}")
    time.sleep(3)  # Wait for page to load

    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.find_all("div", class_="card fund team")

    for card in cards:
        links = card.find_all("a", href=True)
        
        # Get company name and CityU URL (last link in card)
        last_link = links[-1] if links else None
        name = last_link.text.strip() if last_link else "No Info Found"
        cityu_url = "https://www.cityu.edu.hk" + last_link["href"] if last_link else "No Info Found"

        # Visit individual company page to get website and email
        website = "No Info Found"
        email = "No Info Found"

        if cityu_url != "No Info Found":
            try:
                driver.get(cityu_url)
                time.sleep(2)
                company_soup = BeautifulSoup(driver.page_source, "html.parser")
                company_links = company_soup.find_all("a", href=True)

                for link in company_links:
                    href = link["href"]
                    if "mailto:" in href and email == "No Info Found":
                        email = href.replace("mailto:", "").strip()
                    elif (href.startswith("http") and 
                          "cityu.edu.hk" not in href and
                          "facebook.com" not in href and
                          "linkedin.com" not in href and
                          "youtube.com" not in href and
                          website == "No Info Found"):
                        website = href.strip()
            except Exception as e:
                print(f"Error visiting {cityu_url}: {e}")

        all_companies.append({
            "Company Name": name,
            "CityU URL": cityu_url,
            "Company Website": website,
            "Email": email
        })

        print(f"  ✓ Company Name: {name} | CityU URL: {cityu_url} | Company Website: {website} | Email: {email}")

    # Go back to listing page for next page
    print(f"Page {page} done. Total so far: {len(all_companies)}")

print(f"\nAll done! Total companies scraped: {len(all_companies)}")

Scraping page 1 of 40...
  ✓ Company Name: 3D printing low-carbon ecological concrete | CityU URL: https://www.cityu.edu.hk/en/hktech300/start-ups/seed-fund-teams/3d-printing-low-carbon-ecological-concrete | Company Website: No Info Found | Email: No Info Found
  ✓ Company Name: 5th Immersiv | CityU URL: https://www.cityu.edu.hk/en/hktech300/start-ups/seed-fund-teams/5th-immersiv | Company Website: No Info Found | Email: martin@bride-union.com
  ✓ Company Name: A-TCMS (Autonomous Tire Condition Monitoring System) | CityU URL: https://www.cityu.edu.hk/en/hktech300/start-ups/seed-fund-teams/a-tcms | Company Website: No Info Found | Email: No Info Found
  ✓ Company Name: A.I. Ambassador (Virtual Receptionist) kiosk solution | CityU URL: https://www.cityu.edu.hk/en/hktech300/start-ups/seed-fund-teams/ai-ambassador | Company Website: https://www.innocorn.com/ | Email: info@innocorn.com
  ✓ Company Name: A3-ml Ltd | CityU URL: https://www.cityu.edu.hk/en/hktech300/start-ups/seed-fund-teams/a

## Close Browser
**Browser quit once scraping is done**

In [17]:
driver.quit()
print("Browser closed.")

Browser closed.


## Save data to Excel

**Export the scraped data into Excel file**

In [18]:
df = pd.DataFrame(all_companies)
filename = "hktech300_startups.xlsx"
df.to_excel(filename, index=False)
print(f"Done! {len(df)} companies saved to {filename} ✅")  # in this case, filename should be "hktech300_startups.xlsx"
df.head(10)  # Preview first 10 rows

Done! 720 companies saved to hktech300_startups.xlsx ✅


,Company Name,CityU URL,Company Website,Email
0,3D printing low-carbon ecological concrete,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,No Info Found
1,5th Immersiv,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,martin@bride-union.com
2,A-TCMS (Autonomous Tire Condition Monitoring S...,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,No Info Found
3,A.I. Ambassador (Virtual Receptionist) kiosk s...,https://www.cityu.edu.hk/en/hktech300/start-up...,https://www.innocorn.com/,info@innocorn.com
4,A3-ml Ltd,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,director@a3-ml.com
5,A3A PayTech Limited,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,No Info Found
6,ABCDYi,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,hello@abcdyi.net
7,AESIR Institute Limited,https://www.cityu.edu.hk/en/hktech300/start-up...,https://www.aesir.hk/seriousgame.html,aesirhkservice@gmail.com
8,AFJ – TutorGPT,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,No Info Found
9,AI Doodle,https://www.cityu.edu.hk/en/hktech300/start-up...,No Info Found,No Info Found
